# Lesson 4: Having Conversations with AI

## Building Memory into Your AI

In this lesson, you'll learn:
- How AI "remembers" context in conversations
- The message history pattern
- How to build multi-turn conversations
- Build a Memory Bot that remembers you!

**Key insight: AI doesn't actually have memory - YOU provide it!**


---

## Part 1: The Memory Problem

### AI Doesn't Actually Remember!

When you chat with ChatGPT on their website, it seems like it remembers your conversation. But here's the truth:

```
┌─────────────────────────────────────────────────────────────────┐
│                   HOW IT LOOKS TO YOU                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  You: "My name is Alex"                                         │
│  AI:  "Nice to meet you, Alex!"                                 │
│                                                                 │
│  You: "What's my name?"                                         │
│  AI:  "Your name is Alex!"   ← Wow, it remembered!              │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                   WHAT ACTUALLY HAPPENS                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Each time you send a message, the ENTIRE conversation         │
│  history is sent to the AI:                                     │
│                                                                 │
│  API Call 1:                                                    │
│  messages = [                                                   │
│      {"role": "user", "content": "My name is Alex"}             │
│  ]                                                              │
│                                                                 │
│  API Call 2:                                                    │
│  messages = [                                                   │
│      {"role": "user", "content": "My name is Alex"},            │
│      {"role": "assistant", "content": "Nice to meet you, Alex!"},│
│      {"role": "user", "content": "What's my name?"}             │
│  ]                                                              │
│                                                                 │
│  AI reads ALL of this, that's how it "remembers"!               │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### The Secret: Message History

You maintain a list of messages and keep adding to it. Each API call includes the full conversation!


---

## Part 2: Setup


In [3]:
# Setup - Run this first!

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()

print("✅ Ready to learn about conversations!")


✅ Ready to learn about conversations!


---

## Part 3: Single Messages vs Conversations

Let's see the difference between our old approach and a conversational approach.


In [4]:
# OLD APPROACH: No memory (each call is independent)

def chat_no_memory(message):
    """Each call is completely independent"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": message}]
    )
    return response.choices[0].message.content

# Let's test it
print("Testing WITHOUT memory:")
print("-" * 40)

print("You: My name is Alex")
response1 = chat_no_memory("My name is Alex")
print(f"AI: {response1}")

print("\nYou: What is my name?")
response2 = chat_no_memory("What is my name?")
print(f"AI: {response2}")

print("\n❌ See? It doesn't remember!")


Testing WITHOUT memory:
----------------------------------------
You: My name is Alex
AI: Nice to meet you, Alex! How can I assist you today?

You: What is my name?
AI: I'm sorry, but I don't have access to personal data about you unless you've shared it in our conversation. How can I assist you today?

❌ See? It doesn't remember!


In [5]:
# NEW APPROACH: With memory (maintain conversation history)

# Start with an empty conversation
conversation_history = []

def chat_with_memory(message):
    """Maintains conversation history for context"""
    
    # Step 1: Add the user's message to history
    conversation_history.append({"role": "user", "content": message})
    
    # Step 2: Send the ENTIRE history to the API
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=conversation_history  # All previous messages included!
    )
    
    # Step 3: Get the AI's response
    ai_message = response.choices[0].message.content
    
    # Step 4: Add AI's response to history too!
    conversation_history.append({"role": "assistant", "content": ai_message})
    
    return ai_message

print("Testing WITH memory:")
print("-" * 40)

print("You: My name is Alex")
response1 = chat_with_memory("My name is Alex")
print(f"AI: {response1}")

print("\nYou: What is my name?")
response2 = chat_with_memory("What is my name?")
print(f"AI: {response2}")

print("\n✅ Now it remembers!")


Testing WITH memory:
----------------------------------------
You: My name is Alex
AI: Nice to meet you, Alex! How can I assist you today?

You: What is my name?
AI: Your name is Alex!

✅ Now it remembers!


In [7]:
# Let's look at what the conversation history looks like now!

print("Current conversation history:")
print("=" * 50)

for i, message in enumerate(conversation_history):
    role = message["role"].upper()
    content = message["content"]
    print(f"{i+1}. [{role}]: {content[:100]}...")  # Truncate long messages


Current conversation history:
1. [USER]: My name is Alex...
2. [ASSISTANT]: Nice to meet you, Alex! How can I assist you today?...
3. [USER]: What is my name?...
4. [ASSISTANT]: Your name is Alex!...


---

## Part 4: The Three Message Roles

There are actually THREE types of messages:

| Role | Who | Purpose |
|------|-----|---------|
| `user` | You (the human) | Your messages and questions |
| `assistant` | The AI | AI's responses |
| `system` | You (the developer) | Instructions that guide AI behavior |

We'll cover `system` messages in depth in Lesson 5. For now, let's focus on user/assistant.


---

## Part 5: Building a Chatbot Class

Let's build a proper reusable chatbot with memory!


In [6]:
# A proper Chatbot class with memory

class Chatbot:
    """A simple chatbot with conversation memory"""
    
    def __init__(self, name="Assistant"):
        """Initialize the chatbot with an empty conversation history"""
        self.name = name
        self.conversation_history = []
    
    def chat(self, message):
        """Send a message and get a response"""
        # Add user message to history
        self.conversation_history.append({
            "role": "user", 
            "content": message
        })
        
        # Call the API with full history
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=self.conversation_history
        )
        
        # Get AI response
        ai_response = response.choices[0].message.content
        
        # Add AI response to history
        self.conversation_history.append({
            "role": "assistant",
            "content": ai_response
        })
        
        return ai_response
    
    def clear_history(self):
        """Start a fresh conversation"""
        self.conversation_history = []
        print(f"🗑️ {self.name}'s memory cleared!")
    
    def show_history(self):
        """Display the conversation so far"""
        print(f"📜 Conversation History ({len(self.conversation_history)} messages):")
        print("-" * 40)
        for msg in self.conversation_history:
            role = "You" if msg["role"] == "user" else self.name
            print(f"{role}: {msg['content'][:100]}...")

print("✅ Chatbot class created!")


✅ Chatbot class created!


In [8]:
# Let's use our chatbot!

bot = Chatbot(name="Buddy")

print("🤖 Chatbot 'Buddy' is ready!")
print("=" * 50)

# Have a conversation
print("\nYou: Hi! My favorite color is blue and I love pizza.")
print(f"Buddy: {bot.chat('Hi! My favorite color is blue and I love pizza.')}")

print("\nYou: What are two things you know about me?")
print(f"Buddy: {bot.chat('What are two things you know about me?')}")


🤖 Chatbot 'Buddy' is ready!

You: Hi! My favorite color is blue and I love pizza.
Buddy: Hi there! Blue is such a beautiful color, and pizza is a classic favorite! Do you have a favorite type of pizza or toppings?

You: What are two things you know about me?
Buddy: Based on what you've shared, I know that your favorite color is blue, and you love pizza. If there's more you'd like to share or discuss, feel free!


In [9]:
# Show the history
bot.show_history()


📜 Conversation History (4 messages):
----------------------------------------
You: Hi! My favorite color is blue and I love pizza....
Buddy: Hi there! Blue is such a beautiful color, and pizza is a classic favorite! Do you have a favorite ty...
You: What are two things you know about me?...
Buddy: Based on what you've shared, I know that your favorite color is blue, and you love pizza. If there's...


---

## Part 6: Project - Build a Memory Bot!

Let's create an interactive chatbot that you can actually talk to!


In [ ]:
# PROJECT: Interactive Memory Bot!

def run_memory_bot():
    """Run an interactive chatbot with memory"""
    
    # Create a new chatbot
    bot = Chatbot(name="MemoryBot")
    
    print("🧠 MEMORY BOT 🧠")
    print("=" * 50)
    print("I'm MemoryBot! I remember everything you tell me!")
    print("Commands:")
    print("  'quit' - Exit the chat")
    print("  'history' - Show our conversation")
    print("  'clear' - Forget everything and start fresh")
    print("=" * 50)
    print()
    
    while True:
        # Get user input
        user_input = input("You: ").strip()
        
        # Check for special commands
        if user_input.lower() == 'quit':
            print("\n👋 Goodbye! Thanks for chatting!")
            break
        elif user_input.lower() == 'history':
            bot.show_history()
            continue
        elif user_input.lower() == 'clear':
            bot.clear_history()
            continue
        elif user_input == '':
            continue
        
        # Get bot response
        response = bot.chat(user_input)
        print(f"MemoryBot: {response}")
        print()

# Uncomment the line below to run the interactive bot!
# run_memory_bot()


### Try the Memory Bot!

Uncomment `run_memory_bot()` in the cell above and run it to start chatting!

Try telling it:
- Your name
- Your favorite things
- Facts about yourself

Then ask it what it remembers about you!
Solmv 

---

## Summary: Key Takeaways

### How AI "Memory" Works

1. **AI doesn't actually remember** - You provide the memory
2. **Message history** - Keep a list of all messages
3. **Full context** - Send the entire history with each API call
4. **Add both sides** - Store both user AND assistant messages

### The Pattern

```python
# 1. Keep a list of messages
history = []

# 2. Add user message
history.append({"role": "user", "content": user_input})

# 3. Send full history to API
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=history  # Full history!
)

# 4. Add AI response to history
history.append({"role": "assistant", "content": ai_response})
```

### What You Built
A **Memory Bot** class that maintains conversation context!

### What's Next?
In **Lesson 5**, you'll learn about **System Prompts** - how to give AI a personality!

**Great job! Move on to Lesson 5 when ready!**
